# 02 · ETL y modelo dimensional (esquema estrella)
**Objetivo:** transformar el archivo plano en un modelo estrella con una tabla de hechos de envíos y cuatro dimensiones (tiempo con día de la semana, zona, transportista y tipo de servicio), calculando los días de tránsito, el retraso frente a la promesa y la marca de entrega a tiempo.

**Entrada:** `datos/envios_courier.csv`
**Salidas:**
- `bd/courier_dw.sqlite` → base de datos del modelo estrella (para las consultas SQL)
- `salida/powerbi/*.csv` → las 5 tablas para importar en Power BI

**Diseño del modelo:** grano = un envío por fila. `dim_tiempo` es una dimensión de rol: se une tres veces (recepción, promesa y entrega).

## E · Extracción

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

RUTA_DATOS = "datos/envios_courier.csv"
RUTA_BD = Path("bd/courier_dw.sqlite")
RUTA_POWERBI = Path("salida/powerbi")
RUTA_BD.parent.mkdir(exist_ok=True)
RUTA_POWERBI.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RUTA_DATOS,
                 parse_dates=["fecha_recepcion", "fecha_promesa", "fecha_entrega"])
print("Registros leídos:", len(df))

Registros leídos: 41952


## T · Transformación
### 1. Dimensión zona

In [2]:
dim_zona = pd.DataFrame({"zona": sorted(df["zona_destino"].unique())})
dim_zona.insert(0, "id_zona", range(1, len(dim_zona) + 1))
tipo = {"Norte": "Urbana", "Centro": "Urbana", "Sur": "Urbana",
        "Valles": "Periurbana", "Rural": "Rural"}
dim_zona["tipo_area"] = dim_zona["zona"].map(tipo)
dim_zona

,id_zona,zona,tipo_area
0,1,Centro,Urbana
1,2,Norte,Urbana
2,3,Rural,Rural
3,4,Sur,Urbana
4,5,Valles,Periurbana


### 2. Dimensión transportista

In [3]:
dim_transportista = pd.DataFrame(
    {"codigo_transportista": sorted(df["id_transportista"].unique())})
dim_transportista.insert(0, "id_transportista",
                         range(1, len(dim_transportista) + 1))
dim_transportista["nombre_transportista"] = (
    "Transportista " + dim_transportista["codigo_transportista"].str[-2:])
dim_transportista

,id_transportista,codigo_transportista,nombre_transportista
0,1,TR01,Transportista 01
1,2,TR02,Transportista 02
2,3,TR03,Transportista 03
3,4,TR04,Transportista 04
4,5,TR05,Transportista 05
5,6,TR06,Transportista 06
6,7,TR07,Transportista 07
7,8,TR08,Transportista 08


### 3. Dimensión tipo de servicio
El costo base es el costo promedio de un envío entregado al primer intento.

In [4]:
base = (df[(df["estado_final"] == "Entregado") & (df["intentos_entrega"] == 1)]
        .groupby("tipo_servicio")["costo_envio"].mean().round(2))

dim_servicio = pd.DataFrame({
    "id_servicio":   [1, 2],
    "tipo_servicio": ["Estándar", "Express"],
    "dias_promesa":  [3, 1],
})
dim_servicio["costo_base_usd"] = dim_servicio["tipo_servicio"].map(base)
dim_servicio

,id_servicio,tipo_servicio,dias_promesa,costo_base_usd
0,1,Estándar,3,3.04
1,2,Express,1,4.65


### 4. Dimensión tiempo (calendario continuo con día de la semana)

In [5]:
inicio = df["fecha_recepcion"].min()
fin = max(df["fecha_promesa"].max(), df["fecha_entrega"].max())
dim_tiempo = pd.DataFrame({"fecha": pd.date_range(inicio, fin, freq="D")})

dias  = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
meses = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
         "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]

dim_tiempo["id_tiempo"]      = dim_tiempo["fecha"].dt.strftime("%Y%m%d").astype(int)
dim_tiempo["anio"]           = dim_tiempo["fecha"].dt.year
dim_tiempo["trimestre"]      = "T" + dim_tiempo["fecha"].dt.quarter.astype(str)
dim_tiempo["num_mes"]        = dim_tiempo["fecha"].dt.month
dim_tiempo["mes"]            = dim_tiempo["num_mes"].map(lambda m: meses[m - 1])
dim_tiempo["semana_iso"]     = dim_tiempo["fecha"].dt.isocalendar().week.astype(int)
dim_tiempo["dia_mes"]        = dim_tiempo["fecha"].dt.day
dim_tiempo["num_dia_semana"] = dim_tiempo["fecha"].dt.dayofweek + 1
dim_tiempo["dia_semana"]     = dim_tiempo["fecha"].dt.dayofweek.map(lambda d: dias[d])
dim_tiempo["es_fin_semana"]  = (dim_tiempo["num_dia_semana"] >= 6).astype(int)
dim_tiempo = dim_tiempo[["id_tiempo"] + [c for c in dim_tiempo.columns if c != "id_tiempo"]]
print("Días en el calendario:", len(dim_tiempo))
dim_tiempo.head(7)

Días en el calendario: 187


,id_tiempo,fecha,anio,trimestre,num_mes,mes,semana_iso,dia_mes,num_dia_semana,dia_semana,es_fin_semana
0,20260102,2026-01-02,2026,T1,1,Enero,1,2,5,Viernes,0
1,20260103,2026-01-03,2026,T1,1,Enero,1,3,6,Sábado,1
2,20260104,2026-01-04,2026,T1,1,Enero,1,4,7,Domingo,1
3,20260105,2026-01-05,2026,T1,1,Enero,2,5,1,Lunes,0
4,20260106,2026-01-06,2026,T1,1,Enero,2,6,2,Martes,0
5,20260107,2026-01-07,2026,T1,1,Enero,2,7,3,Miércoles,0
6,20260108,2026-01-08,2026,T1,1,Enero,2,8,4,Jueves,0


### 5. Integridad: todas las claves del archivo existen en su dimensión

In [6]:
print("Zonas sin dimensión:", set(df["zona_destino"]) - set(dim_zona["zona"]))
print("Transportistas sin dimensión:",
      set(df["id_transportista"]) - set(dim_transportista["codigo_transportista"]))
print("Servicios sin dimensión:",
      set(df["tipo_servicio"]) - set(dim_servicio["tipo_servicio"]))
fechas_df = pd.concat([df["fecha_recepcion"], df["fecha_promesa"],
                       df["fecha_entrega"].dropna()])
print("Fechas sin dimensión:", (~fechas_df.isin(dim_tiempo["fecha"])).sum())

Zonas sin dimensión: set()
Transportistas sin dimensión: set()


Servicios sin dimensión: set()
Fechas sin dimensión: 0


### 6. Limpieza: imputar el peso inválido (0 kg)

In [7]:
datos = df.copy()
invalido = datos["peso_kg"] <= 0
mediana = datos[~invalido].groupby("tipo_servicio")["peso_kg"].median()
datos["peso_imputado"] = invalido.astype(int)
datos.loc[invalido, "peso_kg"] = datos.loc[invalido, "tipo_servicio"].map(mediana)
print("Pesos imputados:", invalido.sum())
datos.loc[invalido, ["id_envio", "tipo_servicio", "peso_kg", "peso_imputado"]]

Pesos imputados: 1


,id_envio,tipo_servicio,peso_kg,peso_imputado
6448,EN006449,Express,1.855,1


### 7. Medidas derivadas
| Medida | Fórmula |
|---|---|
| `dias_transito` | entrega − recepción |
| `dias_retraso` | entrega − promesa (negativo = antes) |
| `flag_a_tiempo` | 1 si se entregó y retraso ≤ 0 |
| `dias_atraso` | máx(retraso, 0) |

In [8]:
entregado = datos["estado_final"] == "Entregado"
datos["dias_transito"]        = (datos["fecha_entrega"] - datos["fecha_recepcion"]).dt.days
datos["dias_retraso"]         = (datos["fecha_entrega"] - datos["fecha_promesa"]).dt.days
datos["flag_a_tiempo"]        = (entregado & (datos["dias_retraso"] <= 0)).astype(int)
datos["dias_atraso"]          = datos["dias_retraso"].clip(lower=0)
datos["flag_entregado"]       = entregado.astype(int)
datos["flag_devuelto"]        = (~entregado).astype(int)
datos["intentos_adicionales"] = datos["intentos_entrega"] - 1
datos[["id_envio", "dias_transito", "dias_retraso", "flag_a_tiempo"]].head()

,id_envio,dias_transito,dias_retraso,flag_a_tiempo
0,EN000001,1.0,0.0,1
1,EN000002,1.0,-2.0,1
2,EN000003,2.0,-1.0,1
3,EN000004,2.0,-1.0,1
4,EN000005,1.0,0.0,1


### 8. Costo unitario de intentos adicionales y devoluciones

In [9]:
m = datos.groupby(["tipo_servicio", "estado_final", "intentos_entrega"])["costo_envio"].mean()
costo_intento, costo_devolucion = {}, {}
for s in ["Estándar", "Express"]:
    b = m[(s, "Entregado", 1)]
    costo_intento[s] = ((m[(s, "Entregado", 2)] - b) + (m[(s, "Entregado", 3)] - b) / 2) / 2
    costo_devolucion[s] = m[(s, "Devuelto", 1)] - b
    print(f"{s:9s} base USD {b:.2f} | por intento extra USD {costo_intento[s]:.2f}"
          f" | por devolución USD {costo_devolucion[s]:.2f}")

datos["sobrecosto_intentos"]   = datos["intentos_adicionales"] * datos["tipo_servicio"].map(costo_intento)
datos["sobrecosto_devolucion"] = datos["flag_devuelto"] * datos["tipo_servicio"].map(costo_devolucion)

Estándar  base USD 3.04 | por intento extra USD 1.49 | por devolución USD 2.02
Express   base USD 4.65 | por intento extra USD 1.51 | por devolución USD 1.93


### 9. Tabla de hechos con claves foráneas

In [10]:
clave_zona = dict(zip(dim_zona["zona"], dim_zona["id_zona"]))
clave_tr   = dict(zip(dim_transportista["codigo_transportista"], dim_transportista["id_transportista"]))
clave_sv   = dict(zip(dim_servicio["tipo_servicio"], dim_servicio["id_servicio"]))

def clave_fecha(serie):
    return serie.dt.strftime("%Y%m%d").astype("Int64")

fact_envios = pd.DataFrame({
    "id_envio":              datos["id_envio"],
    "id_tiempo_recepcion":   clave_fecha(datos["fecha_recepcion"]),
    "id_tiempo_promesa":     clave_fecha(datos["fecha_promesa"]),
    "id_tiempo_entrega":     clave_fecha(datos["fecha_entrega"]),
    "id_zona":               datos["zona_destino"].map(clave_zona),
    "id_transportista":      datos["id_transportista"].map(clave_tr),
    "id_servicio":           datos["tipo_servicio"].map(clave_sv),
    "estado_final":          datos["estado_final"],
    "peso_kg":               datos["peso_kg"],
    "peso_imputado":         datos["peso_imputado"],
    "intentos_entrega":      datos["intentos_entrega"],
    "intentos_adicionales":  datos["intentos_adicionales"],
    "costo_envio":           datos["costo_envio"],
    "dias_transito":         datos["dias_transito"].astype("Int64"),
    "dias_retraso":          datos["dias_retraso"].astype("Int64"),
    "dias_atraso":           datos["dias_atraso"].astype("Int64"),
    "flag_entregado":        datos["flag_entregado"],
    "flag_devuelto":         datos["flag_devuelto"],
    "flag_a_tiempo":         datos["flag_a_tiempo"],
    "sobrecosto_intentos":   datos["sobrecosto_intentos"].round(4),
    "sobrecosto_devolucion": datos["sobrecosto_devolucion"].round(4),
})
print(fact_envios.shape)
fact_envios.head()

(41952, 21)


,id_envio,id_tiempo_recepcion,id_tiempo_promesa,id_tiempo_entrega,id_zona,id_transportista,id_servicio,estado_final,peso_kg,peso_imputado,...,intentos_adicionales,costo_envio,dias_transito,dias_retraso,dias_atraso,flag_entregado,flag_devuelto,flag_a_tiempo,sobrecosto_intentos,sobrecosto_devolucion
0,EN000001,20260102,20260103,20260103,2,5,2,Entregado,0.72,0,...,0,4.25,1,0,0,1,0,1,0.0,0.0
1,EN000002,20260102,20260105,20260103,2,2,1,Entregado,2.31,0,...,0,2.89,1,-2,0,1,0,1,0.0,0.0
2,EN000003,20260102,20260105,20260104,4,5,1,Entregado,1.66,0,...,0,2.93,2,-1,0,1,0,1,0.0,0.0
3,EN000004,20260102,20260105,20260104,2,7,1,Entregado,0.26,0,...,0,3.41,2,-1,0,1,0,1,0.0,0.0
4,EN000005,20260102,20260103,20260103,1,8,2,Entregado,1.61,0,...,0,4.29,1,0,0,1,0,1,0.0,0.0


### 10. Validación de la tabla de hechos

In [11]:
print("Claves vacías:", fact_envios[["id_zona", "id_transportista", "id_servicio"]].isna().sum().sum())
print("Fechas de entrega vacías (= devoluciones):", fact_envios["id_tiempo_entrega"].isna().sum())
print("IDs únicos:", fact_envios["id_envio"].is_unique)
otd = fact_envios["flag_a_tiempo"].sum() / fact_envios["flag_entregado"].sum()
print(f"OTD global (control): {otd:.2%}")

Claves vacías: 0
Fechas de entrega vacías (= devoluciones): 713
IDs únicos: True
OTD global (control): 74.52%


## L · Carga
Se guarda el modelo en SQLite (para SQL) y en CSV (para Power BI).

In [12]:
tablas = {"dim_tiempo": dim_tiempo, "dim_zona": dim_zona,
          "dim_transportista": dim_transportista, "dim_servicio": dim_servicio,
          "fact_envios": fact_envios}

with sqlite3.connect(RUTA_BD) as con:
    for nombre, tabla in tablas.items():
        t = tabla.copy()
        if nombre == "dim_tiempo":
            t["fecha"] = t["fecha"].dt.strftime("%Y-%m-%d")
        t.to_sql(nombre, con, if_exists="replace", index=False)

for nombre, tabla in tablas.items():
    tabla.to_csv(RUTA_POWERBI / f"{nombre}.csv", index=False,
                 date_format="%Y-%m-%d", encoding="utf-8-sig")

for nombre, tabla in tablas.items():
    print(f"{nombre:20s} {len(tabla):6d} filas")
print("\nBase de datos:", RUTA_BD)
print("CSV para Power BI:", RUTA_POWERBI)

dim_tiempo              187 filas
dim_zona                  5 filas
dim_transportista         8 filas
dim_servicio              2 filas
fact_envios           41952 filas

Base de datos: bd/courier_dw.sqlite
CSV para Power BI: salida/powerbi
